# Code Optimization Summary (juhoon branch merge)

이 문서는 `sunkim` 브랜치에 `juhoon` 브랜치의 성능 개선 사항을 병합(merge)한 내역을 설명합니다.
주로 `rk3.cuh`와 `weno5.cuh`에서 메모리 적정성 향상 및 명령어(Instruction) 최소화 기법이 적용되었습니다.

## 1. `rk3.cuh` 최적화 내역

### 1.1 Local Variable Caching (레지스터 최적화)
기존에는 `params.nx_total`, `params.ny_total` 등 구조체의 멤버 변수에 매번 접근하여 그리드 인덱스를 넘겨주었습니다.
이를 지역 상수(Local constant)로 캐싱하여 메모리 접근 횟수를 줄이고 성능을 개선했습니다.

```cpp
// 변경 전
int index = idx(i, j, k, params.nx_total, params.ny_total);

// 변경 후
const int nx_tot = params.nx_total;
const int ny_tot = params.ny_total;
const int index = idx(i, j, k, nx_tot, ny_tot);
```

### 1.2 Syntax 정리
변수 초기화 방식을 `double u_eff{...}` 형태에서 호환성을 고려하여 `double u_eff = ...` 형태로 변경하였습니다.

## 2. `weno5.cuh` 최적화 내역

WENO5 알고리즘은 많은 연산을 동반하므로 CUDA 내장 함수를 적극적으로 활용하여 명령어 레벨 최적화를 수행했습니다.

### 2.1 Pointer `__restrict__` 키워드 추가
GPU 커널 함수 매개변수에 `__restrict__`를 사용하여 포인터 앨리어싱(Pointer Aliasing)이 없음을 컴파일러에게 보장합니다. 이는 컴파일러가 더 많은 연산을 최적화하고 레지스터/캐시를 효율적으로 사용할 수 있게 돕습니다.

```cpp
__device__ inline double weno5_left(const double* __restrict__ v) 
```

### 2.2 Fused Multiply-Add (`__fma_rn`) 활용
비선형 가중치(non-linear weights)를 계산할 때, 단항 곱셈 및 덧셈을 한 개의 하드웨어 사이클 안에서 처리하는 내장 함수인 `__fma_rn(x, y, z) = x * y + z`를 사용했습니다.
상수 `epsilon`의 제곱합을 `__fma_rn`과 결합하여 레지스터 사이클을 아꼈습니다.

```cpp
// 변경 후
constexpr double eps_sq{WENO_EPSILON * WENO_EPSILON};
double alpha0 = 0.1 / __fma_rn(beta[0], beta[0], eps_sq);
```

### 2.3 Division 최적화 (나눗셈 연산 감축)
가중치를 정규화할 때 사용되는 나눗셈을 최소화했습니다. GPU에서 나눗셈(`/`)은 곱셈(`*`)보다 연산 비용이 월등히 높습니다.

```cpp
// 변경 전 (나눗셈 3회)
double sum_alpha = alpha0 + alpha1 + alpha2;
double omega0 = alpha0 / sum_alpha;
double omega1 = alpha1 / sum_alpha;
double omega2 = alpha2 / sum_alpha;

// 변경 후 (나눗셈 1회, 곱셈 3회)
double sum_alpha_inv = 1.0 / (alpha0 + alpha1 + alpha2);
double omega0 = alpha0 * sum_alpha_inv;
double omega1 = alpha1 * sum_alpha_inv;
double omega2 = alpha2 * sum_alpha_inv;
```

### 2.4 Polynomial 나눗셈 위치 정리
후보 다항식(`p0`, `p1`, `p2`)을 계산할 때 각각 `6.0`으로 나누던 것을 하나로 묶어, 가중치 조합이 끝난 뒤 최종 반환값에서 한 번만 `6.0`으로 나누도록 `factorization`을 거쳐 부동소수점 곱셈/나눗셈을 아꼈습니다.

```cpp
// 변경 후 -> 마지막에 한 번만 나눗셈 수행
return (omega0 * p0 + omega1 * p1 + omega2 * p2) / 6.0;
```